# FRED Macro Briefing Crew

This notebook demonstrates a three-agent CrewAI workflow that uses four FRED macroeconomic series:

- Consumer Price Index (`CPIAUCSL`)
- Unemployment Rate (`UNRATE`)
- Federal Funds Rate (`FEDFUNDS`)
- 10-Year Minus 2-Year Treasury Spread (`T10Y2Y`)

The Researcher collects and profiles the data, the Analyst calculates deterministic statistics, and the Writer produces a concise macroeconomic briefing.

In [ ]:
from pathlib import Path
import json
import runpy

import pandas as pd
from IPython.display import Image, Markdown, display

import fred_tools as ft
import reporting

## Load the saved FRED snapshot

Use the dated CSV snapshot so that the analysis remains reproducible.

In [ ]:
SNAPSHOT_DIR = Path("data/fred/2026-11-04")

if not SNAPSHOT_DIR.exists():
    raise FileNotFoundError(
        f"FRED snapshot does not exist: {SNAPSHOT_DIR}"
    )

profiles = ft.profile_fred_snapshot(SNAPSHOT_DIR)

profile_table = pd.DataFrame(profiles).T[
    [
        "series_name",
        "latest_date",
        "latest_value",
        "prior_year_date",
        "prior_year_value",
        "missing_values",
    ]
]
display(profile_table)

## Calculate deterministic macroeconomic statistics

The Analyst tool uses pandas-based calculations rather than asking the language model to perform arithmetic.

In [ ]:
statistics = ft.calculate_macro_statistics(SNAPSHOT_DIR)

statistics_table = pd.DataFrame(
    {
        "Statistic": statistics.keys(),
        "Value": statistics.values(),
    }
)
display(statistics_table)

## Visualize the four FRED series

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHART_PATH = OUTPUT_DIR / "fred_macro_series.png"
reporting.create_fred_chart(
    SNAPSHOT_DIR,
    CHART_PATH,
)

display(Image(filename=str(CHART_PATH)))

## Review the generated macroeconomic briefing

In [ ]:
BRIEFING_PATH = (
    OUTPUT_DIR / "fred_macro_briefing_2026-11-04.md"
)

if BRIEFING_PATH.exists():
    display(
        Markdown(
            BRIEFING_PATH.read_text(encoding="utf-8")
        )
    )
else:
    print(
        "Run crewai.example.py to generate the briefing."
    )

## Compare the CrewAI workflow with the single-LLM baseline

In [ ]:
EVALUATION_PATH = OUTPUT_DIR / "evaluation_report.json"

if EVALUATION_PATH.exists():
    evaluation = json.loads(
        EVALUATION_PATH.read_text(encoding="utf-8")
    )
    comparison = pd.DataFrame(
        {
            "Method": [
                "Three-agent CrewAI workflow",
                "Single-LLM baseline",
            ],
            "Correct claims": [
                evaluation["crew"]["correct_claims"],
                evaluation["single_llm_baseline"][
                    "correct_claims"
                ],
            ],
            "Total claims": [
                evaluation["crew"]["total_claims"],
                evaluation["single_llm_baseline"][
                    "total_claims"
                ],
            ],
            "Fraction correct": [
                evaluation["crew"]["fraction_correct"],
                evaluation["single_llm_baseline"][
                    "fraction_correct"
                ],
            ],
        }
    )
    display(comparison)
else:
    print("Run evaluation.py to generate the comparison.")

## Optionally run the three-agent crew

LLM execution is disabled by default so the notebook can be tested without making paid or rate-limited API calls. Set `RUN_LLM = True` only when a Gemini API key and `CREWAI_MODEL` are available.

In [ ]:
RUN_LLM = False

if RUN_LLM:
    example_module = runpy.run_path(
        "crewai.example.py",
        run_name="crewai_example",
    )
    result = example_module["run_fred_crew"](
        data_dir="data/fred",
        download_date="2026-11-04",
        llm=None,
        verbose=False,
    )
    display(Markdown(str(result)))
else:
    print(
        "LLM execution is disabled. Set RUN_LLM = True "
        "to run the three-agent crew."
    )